# 📈 Exp D: Causal Uplift Modeling & Double Machine Learning (`T-Learner`, `X-Learner` & `Double/Debiased ML`) for Campaign ROI Optimization
**Author:** Akmal Khan, PhD  
**Repository:** [github.com/akmalkhaniub/enterprise-banking-ds-platform](https://github.com/akmalkhaniub/enterprise-banking-ds-platform)

### 🎯 Why Causal Inference Instead of Standard Churn Prediction?
Standard propensity models target **Sure Things** (customers who would renew anyway, wasting promotional discounts) and **Sleeping Dogs** (customers who churn *because* a marketing nudge reminds them to cancel). Here we estimate **Conditional Average Treatment Effects ($\tau(X) = E[Y(1) - Y(0) | X]$)** using:
1. **S-Learner** (Single Model with Treatment Indicator)
2. **T-Learner** (Two Separate Gradient Boosted Trees)
3. **X-Learner** (Cross-Imputed Counterfactual Meta-Learner)
4. **Double / Debiased Machine Learning (Chernozhukov et al. Orthogonalized Residual-on-Residual)**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 15000
# Customer behavioral features
price_sensitivity = np.random.uniform(-1.5, 1.5, size=n)
engagement_score = np.random.uniform(-1.5, 1.5, size=n)
tenure_norm = np.random.normal(0, 1, size=n)
balance_tier = np.random.normal(0, 1, size=n)
X = np.column_stack([price_sensitivity, engagement_score, tenure_norm, balance_tier])

# Confounded historical treatment assignment (propensity score e(X))
propensity_logit = 0.65 * engagement_score - 0.45 * price_sensitivity
e_x = 1.0 / (1.0 + np.exp(-propensity_logit))
T = (np.random.uniform(0, 1, size=n) < e_x).astype(int)

# True Heterogeneous Treatment Effect tau(X):
# - Persuavables (high price sensitivity, moderate engagement) -> strong positive uplift (+0.32)
# - Sleeping Dogs (low price sensitivity, high negative annoyance) -> negative uplift (-0.18)
true_tau = 0.22 * price_sensitivity - 0.12 * (engagement_score ** 2) + 0.08 * np.sin(tenure_norm)
base_outcome = 0.45 + 0.18 * engagement_score + 0.10 * balance_tier
y_prob = np.clip(base_outcome + T * true_tau, 0.02, 0.98)
Y = (np.random.uniform(0, 1, size=n) < y_prob).astype(float)

X_tr, X_te, T_tr, T_te, Y_tr, Y_te, tau_tr, tau_te = train_test_split(
    X, T, Y, true_tau, test_size=0.30, random_state=42
)
print(f'Train: {len(X_tr):,} | Test: {len(X_te):,} | Avg True Treatment Effect: {true_tau.mean():.4f}')

In [ ]:
# 1. S-Learner, 2. T-Learner, 3. X-Learner, and 4. Double/Debiased ML (Orthogonalized R-Learner)
# S-Learner
s_model = GradientBoostingRegressor(n_estimators=150, max_depth=4, random_state=42)
s_model.fit(np.column_stack([X_tr, T_tr]), Y_tr)
cate_s = s_model.predict(np.column_stack([X_te, np.ones(len(X_te))])) - s_model.predict(np.column_stack([X_te, np.zeros(len(X_te))]))

# T-Learner
m0 = GradientBoostingRegressor(n_estimators=150, max_depth=4, random_state=42).fit(X_tr[T_tr == 0], Y_tr[T_tr == 0])
m1 = GradientBoostingRegressor(n_estimators=150, max_depth=4, random_state=42).fit(X_tr[T_tr == 1], Y_tr[T_tr == 1])
cate_t = m1.predict(X_te) - m0.predict(X_te)

# X-Learner (Imputed treatment effects D1 and D0 weighted by propensity score)
d1 = Y_tr[T_tr == 1] - m0.predict(X_tr[T_tr == 1])
d0 = m1.predict(X_tr[T_tr == 0]) - Y_tr[T_tr == 0]
tau1 = GradientBoostingRegressor(n_estimators=120, max_depth=3, random_state=42).fit(X_tr[T_tr == 1], d1)
tau0 = GradientBoostingRegressor(n_estimators=120, max_depth=3, random_state=42).fit(X_tr[T_tr == 0], d0)
prop_model = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X_tr, T_tr)
g_te = np.clip(prop_model.predict_proba(X_te)[:, 1], 0.05, 0.95)
cate_x = g_te * tau0.predict(X_te) + (1.0 - g_te) * tau1.predict(X_te)

# Double / Debiased ML (Robinson / Chernozhukov Orthogonalization)
y_hat_nuisance = GradientBoostingRegressor(n_estimators=120, max_depth=3, random_state=42).fit(X_tr, Y_tr).predict(X_tr)
t_hat_nuisance = np.clip(prop_model.predict_proba(X_tr)[:, 1], 0.05, 0.95)
y_res = Y_tr - y_hat_nuisance
t_res = T_tr - t_hat_nuisance
pseudo_outcome = y_res / np.where(np.abs(t_res) < 0.05, np.sign(t_res) * 0.05 + 1e-4, t_res)
weights = t_res ** 2
dml_model = GradientBoostingRegressor(n_estimators=150, max_depth=3, random_state=42)
dml_model.fit(X_tr, pseudo_outcome, sample_weight=weights)
cate_dml = dml_model.predict(X_te)

In [ ]:
# Evaluate PEHE (Precision in Estimation of Heterogeneous Effect), Rank Correlation & Top-30% Policy ROI Gain
def eval_causal(name, cate_pred):
    pehe = np.sqrt(np.mean((tau_te - cate_pred) ** 2))
    corr = np.corrcoef(tau_te, cate_pred)[0, 1]
    # Simulate targeting only the top 30% customers ranked by predicted uplift
    top_30_mask = cate_pred >= np.percentile(cate_pred, 70.0)
    incremental_conversions_per_1k = tau_te[top_30_mask].mean() * 1000.0
    avoided_sleeping_dogs_pct = (tau_te[cate_pred < 0] < 0).mean() * 100.0 if (cate_pred < 0).any() else 0.0
    return {
        'Causal Estimator': name,
        'Sqrt-PEHE Error (Lower=Better)': round(float(pehe), 4),
        'CATE Rank Correlation': round(float(corr), 4),
        'Top-30% Policy Lift (per 1k Targeted)': f'+{incremental_conversions_per_1k:.1f}',
        'Sleeping-Dog Detection Precision (%)': round(float(avoided_sleeping_dogs_pct), 1)
    }

random_score = np.random.uniform(-0.1, 0.1, size=len(tau_te))
causal_df = pd.DataFrame([
    eval_causal('0. Random Targeting Baseline (No Uplift Model)', random_score),
    eval_causal('1. S-Learner (Single GBR with Treatment Flag)', cate_s),
    eval_causal('2. T-Learner (Two-Model Counterfactual GBR)', cate_t),
    eval_causal('3. X-Learner (Propensity-Weighted Cross-Imputation)', cate_x),
    eval_causal('4. Double / Debiased ML (Orthogonalized R-Learner)', cate_dml),
]).sort_values('Sqrt-PEHE Error (Lower=Better)').reset_index(drop=True)

print('=== CAUSAL UPLIFT & DOUBLE MACHINE LEARNING BENCHMARK ===')
print(causal_df.to_string(index=False))